# PSELDNets — outdoor_siren_v7 学習（train 240 / val 80、地面反射ON=フル物理版）

**v7 = v6と完全同一の320シーン（同一シード）で、地面反射だけをONにした版**:
対象音・妨害車の両方に two-ray（像音源、剛体地面 R=+1）の地面反射を適用。
速度・距離・SNR・SIR・発音区間・クラス構成はv6と同一。ラベルは従来どおり
**直接音の方向**（反射で汚れた音から真の音源方向を答えるタスク）。

**位置づけ**: 物理ablation（ドップラー/大気吸収/1・r/地面反射を1要素ずつ外す）の
**基準（フル物理）土俵**。fastsimに実装済みの物理スイッチはこのv7から外す方向で使う
（ablation本体はゼミ合意後）。v7は「完全版」ではなく、はしごの次の段
（この先: ポリフォニー・歩行マイク・非剛体地面・実録検証セット等が候補）。

**物理の事前実測（v6クリップで確認済み）**: 地面反射はIV仰角を中央値1.4〜3.3°
押し下げるが方位は不変（0.3°のまま）→ LE/仰角系の難化が予想される。

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
2. **Drive の `MyDrive/PSELDNets_data/` に `dataset_outdoor_siren_v7.zip` をアップロード済み**であること
3. セルを**上から順に**実行（途中でスキップしない）
4. 中断しても再実行すれば続きから学習が再開される（Drive永続化＋自動resume）
5. **注意**: データを差し替えて再学習するときは、必ず下の`EXP_NAME`を新しい名前に変える
   （固定experiment_name使い回しのNaN崩壊事故を参照）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントとパス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定（必要ならここだけ書き換える） ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'   # zip の置き場所
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'   # 学習ログ・ckpt の永続化先
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'  # 事前学習ckptのキャッシュ
DATASET    = 'outdoor_siren_v7'
EXP_NAME   = 'outdoor_siren_v7_run1'                             # 固定名（resume用）
ZIP_NAME   = f'dataset_{DATASET}.zip'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
ZIP_PATH = f'{DRIVE_DATA}/{ZIP_NAME}'
assert os.path.exists(ZIP_PATH), f'⚠️ {ZIP_PATH} がありません。zip をアップロードしてください。'
print(f'OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（v1-v4 と同一・再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive へキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開

zip は `datasets/...` 構成なのでリポジトリ直下で解凍するだけ。
クラス辞書 `cls_indices_train.tsv`（**本データセット専用の4クラス**: Siren/Horn/
BackupBeep/BikeBell）も同梱。

In [ ]:
import zipfile

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('.')
    print('解凍完了')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 320 and n_meta == 320 and n_cls == 4, '⚠️ ファイル数が想定と違います'

## 7. 設定ファイル2つを作成（新規追加のみ・リポジトリ既存ファイルは無編集）

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v7: [fold1_room1]
valid_dataset:
  outdoor_siren_v7: [fold2_room1]
test_dataset:
  outdoor_siren_v7: [fold2_room1]
"""

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v7.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v7

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

open('configs/data/outdoor_siren_v7.yaml', 'w').write(data_yaml)
open('configs/experiment/outdoor_siren_v7.yaml', 'w').write(exp_yaml)
print('wrote configs/data/outdoor_siren_v7.yaml')
print('wrote configs/experiment/outdoor_siren_v7.yaml')

## 8. 前処理（ラベル → HDF5、クリップ索引の作成。1分未満）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('既に前処理済み')
!head -3 {IDX}

## 9. 実行前チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     'チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (4クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA データ (320)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (320)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'クリップ索引'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 60〜90 分見込み、100epoch）

- ログと ckpt は Drive（`PSELDNets_logs`）に直接書くので、セッションが切れても消えない
- `last.ckpt` があれば自動で続きから再開（固定 experiment_name 方式）
- **データを差し替えて再学習する場合は、上のcell4で`EXP_NAME`を新しい名前
  （`outdoor_siren_v7_run2`等）に変えてから実行すること**
- epoch数100はv6 run1と同一（train240本、v6はep45以降収束・ベストep75/85だった）
- メモリ不足になったら `model.batch_size=4` に下げて再実行

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 結果の確認

`val/macro` の ER / F / LE / LR / **SELD_scr** を全エポック分抽出する。
**この val は学習に見せていない80本**（v6と同一シーンの反射込み音声）。

**v7の見どころ（比較基準はv6 run1: ER 0.044 / F 97.0% / LE 4.8° / SELD_scr 0.028）**:
- 反射込みでも同水準 → 反射はモデルにとって難しくない（ablationでも反射offの寄与は小と予想）
- LE/ERが悪化 → 反射がこの土俵の新しい難所（仰角バイアスをどこまで見抜けるか）
- v6とv7は**同一シーン分割**なので、v6モデル↔v7モデルを相互のvalで評価すると
  「反射を知らないモデルは反射入り音声でどれだけ崩れるか」（sim-to-realの縮図）が測れる

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---
## メモ

- データ生成条件の全記録はローカルの `outdoor_seld_e2e/out/dataset_outdoor_siren_v7/`
  （`inspection.csv`=検品結果。v7は反射の物理により仰角ゲートのみ5.0°に緩和、方位は2.0°のまま）
- v6との違いは**地面反射のみ**（two-ray像音源・R=+1、対象音と妨害車の両方）。
  シーン・シード・件数・クラス構成・SNR/SIRレンジはv6と完全同一（`--v7`フラグ）
- ラベルは直接音の放射時刻DOA（反射は入力側の外乱、教師は真の方向）
- **v6とのクロス評価が可能**: v6/v7はシーン分割が同一（train=idx0-239, val=idx240-319）
  なので、v6モデルをv7のvalで・v7モデルをv6のvalで評価しても学習リークはない
- **experiment_nameの使い回しに注意**（データを変えたら必ず新しい名前にする）
- 学習後の誤り解剖: infer(mode=test) → 予測CSVを連結（`infer_*_all.csv`方式）→
  `scripts/step8_error_anatomy_mc.py --pred ... --ds out/dataset_outdoor_siren_v7`